# Causality violation — the third great PINN pathology

A PINN trains on **all of $[0,T]$ at once**, but physics flows *forward in time* from the
initial condition. Gradient descent happily minimises the residual at late times first —
with a wrong solution — before the IC has had any say. This notebook shows the cleanest
known example, and the few-line fix (**causal weighting**, Wang et al. 2022).

**The toy problem** — a reaction equation (Krishnapriyan et al.'s PINN failure mode):
$$u_t = \rho\,u(1-u),\qquad u(x,0) = e^{-(x-\pi)^2/2(\pi/4)^2},\qquad x\in[0,2\pi],\ t\in[0,1],\ \rho=10$$
$$u_{exact}(x,t) = \frac{h(x)e^{\rho t}}{h(x)e^{\rho t} + 1 - h(x)} \quad\text{(logistic growth at each } x\text{)}$$

**The trap:** $u\equiv 0$ satisfies the PDE **exactly** (it's an equilibrium — just an
unstable one with the wrong IC). The plain PINN takes the bait: it trades a small IC penalty
near $t=0$ for a *perfectly zero* PDE residual everywhere else, and collapses to the trivial
solution. Watching the error **increase** during training is the tell.

**The fix — causal weighting:** split time into $M$ bins with losses $L_1..L_M$ and weight
$$w_k = \exp\Big(-\varepsilon \sum_{j<k} L_j\Big)$$
Late bins are silenced until earlier bins have converged — training sweeps left-to-right
through time, like a time-stepper, without ever building one.

Runs in ~1–2 minutes on Colab (CPU or GPU).

In [ ]:
# Cell 1 -- Problem, exact solution, data
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

RHO = 10.0
def h0(x): return torch.exp(-(x-np.pi)**2/(2*(np.pi/4)**2))
def u_exact(x, t):
    h = h0(x); E = torch.exp(RHO*t)
    return h*E/(h*E + 1 - h)

# evaluation grid (for error + plots)
xe = torch.linspace(0, 2*np.pi, 101, device=device)
te = torch.linspace(0, 1, 101, device=device)
XX, TT = torch.meshgrid(xe, te, indexing='ij')
Pe = torch.stack([XX.ravel(), TT.ravel()], 1)
Ue = u_exact(Pe[:, 0:1], Pe[:, 1:2])

# FIXED collocation points, stratified into M time bins (needed for causal weights)
M, Nb = 20, 100
tb = (torch.arange(M).repeat_interleave(Nb).float() + torch.rand(M*Nb)) / M
xb = torch.rand(M*Nb) * 2*np.pi
P  = torch.stack([xb, tb], 1).to(device)
bin_id = (tb*M).long().clamp(max=M-1).to(device)

x_ic = torch.linspace(0, 2*np.pi, 256, device=device).reshape(-1, 1)
u_ic = h0(x_ic)

def make_mlp():
    torch.manual_seed(0)   # identical init for a fair comparison
    return nn.Sequential(nn.Linear(2, 64), nn.Tanh(),
                         nn.Linear(64, 64), nn.Tanh(),
                         nn.Linear(64, 64), nn.Tanh(),
                         nn.Linear(64, 1)).to(device)

## The theory: why the optimizer picks the impostor

**A PDE residual does not have a unique zero.** The equation $u_t = \rho u(1-u)$ is
satisfied by an entire *family* of solutions — one per initial condition — including two
constants: $u\equiv 0$ (unstable equilibrium) and $u\equiv 1$ (stable equilibrium). Both
make the residual **exactly zero at every collocation point**. The only term in the whole
loss that distinguishes the true solution from these impostors is the IC at $t=0$.

**The loss has no arrow of time.** Physics propagates information forward from the IC; the
PINN loss treats $t=0.9$ and $t=0.1$ as equals. Trace what gradient descent does:

1. The network initialises near $u\approx 0$ → near-zero residual *everywhere* — a deep,
   wide basin, ready-made.
2. The IC loss pulls $u(x,0)$ up toward the Gaussian; near $t=0$ the network complies.
3. But threading the true solution's steep transition ($e^{\rho t}$ at $\rho=10$) requires
   passing through configurations with **large residuals at intermediate times**. The mean
   loss punishes the journey.
4. So the optimizer takes the cheap deal: satisfy the IC in a thin sliver near $t=0$, relax
   back to the exact-but-wrong $u\equiv 0$ everywhere else.

The tell: **the loss falls while the error grows** (0.91 → 0.99 in the run below). The
optimizer is succeeding at the objective it was given; the objective is what's broken.

**Two cures, one idea — force training to consume time in the order physics does:**
- **Causal weighting (soft):** within one training run, gate late-time bins on the
  convergence of earlier ones. Implemented in Cell 2.
- **Time-marching + transfer learning (hard):** split $[0,T]$ into windows, solve them in
  sequence, hand each window's end state to the next as its IC, and **warm-start each
  window's network from the previous window's weights**. Implemented after the main
  comparison below.

In [ ]:
# Cell 2 -- One training loop with a `causal` switch
def train(causal, eps=100.0, epochs=12000, tag=''):
    model = make_mlp()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    hist = {'epoch': [], 'err': []}
    w_snaps = {}
    t0 = time.perf_counter()
    for e in range(epochs):
        opt.zero_grad()
        Pc = P.detach().clone().requires_grad_(True)
        u  = model(Pc)
        g  = torch.autograd.grad(u, Pc, torch.ones_like(u), create_graph=True)[0]
        r2 = (g[:, 1:2] - RHO*u*(1-u))**2          # residual^2 of u_t = rho u(1-u)
        if causal:
            # --- causal weighting: silence bin k until bins < k have converged ---
            Lb = torch.stack([r2[bin_id == k].mean() for k in range(M)])
            with torch.no_grad():
                w = torch.exp(-eps*torch.cumsum(
                        torch.cat([torch.zeros(1, device=device), Lb[:-1].detach()]), 0))
            loss_pde = (w*Lb).sum() / w.sum()
            if e in (0, 1000, 4000, 11999): w_snaps[e] = w.cpu().numpy()
        else:
            loss_pde = r2.mean()
        loss_ic = ((model(torch.cat([x_ic, torch.zeros_like(x_ic)], 1)) - u_ic)**2).mean()
        (loss_pde + 100*loss_ic).backward(); opt.step()
        if e % 200 == 0 or e == epochs-1:
            with torch.no_grad():
                err = torch.sqrt(torch.mean((model(Pe)-Ue)**2)/torch.mean(Ue**2)).item()
            hist['epoch'].append(e); hist['err'].append(err)
    if device.type == 'cuda': torch.cuda.synchronize()
    print(f'{tag}: {time.perf_counter()-t0:.0f} s | final relative L2 error = {hist["err"][-1]:.4f}')
    return model, hist, w_snaps

model_plain,  hist_plain,  _      = train(causal=False, tag='plain PINN ')
model_causal, hist_causal, wsnaps = train(causal=True,  tag='causal PINN')

In [ ]:
# Cell 3 -- The collapse, the cure, and the moving training frontier
fig, ax = plt.subplots(1, 3, figsize=(15.5, 4))

# (a) profiles at t = 0, 0.5, 1
xs = torch.linspace(0, 2*np.pi, 200, device=device).reshape(-1, 1)
for t, ls in [(0.0, ':'), (0.5, '--'), (1.0, '-')]:
    tt = torch.full_like(xs, t)
    ax[0].plot(xs.cpu(), u_exact(xs, tt).cpu(), 'g'+ls, lw=1.8, alpha=.8)
    with torch.no_grad():
        ax[0].plot(xs.cpu(), model_plain(torch.cat([xs, tt], 1)).cpu(),  'b'+ls, lw=1.4)
        ax[0].plot(xs.cpu(), model_causal(torch.cat([xs, tt], 1)).cpu(), 'r'+ls, lw=1.4)
ax[0].plot([], [], 'g-', label='exact'); ax[0].plot([], [], 'b-', label='plain (collapsed to 0)')
ax[0].plot([], [], 'r-', label='causal'); ax[0].set_title('t = 0 (dotted), 0.5 (dashed), 1 (solid)')
ax[0].set_xlabel('x'); ax[0].set_ylabel('u'); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

# (b) relative error vs epoch — plain gets WORSE
ax[1].semilogy(hist_plain['epoch'],  hist_plain['err'],  'b', label='plain')
ax[1].semilogy(hist_causal['epoch'], hist_causal['err'], 'r', label='causal')
ax[1].set_title('Plain error INCREASES while its loss decreases')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('relative L2 error'); ax[1].legend(); ax[1].grid(alpha=.3, which='both')

# (c) causal weights sweep forward through time like a solver
tcent = (np.arange(M)+0.5)/M
for e, w in sorted(wsnaps.items()):
    ax[2].plot(tcent, w, 'o-', ms=3, label=f'epoch {e}')
ax[2].set_title('Causal weights: a training frontier moving forward in t')
ax[2].set_xlabel('time bin'); ax[2].set_ylabel('weight  $w_k$'); ax[2].legend(fontsize=8); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Remedy 2 — time-marching with transfer learning

Causal weighting is the *soft* enforcement of time-ordering inside one training run.
The *hard* version — what scales to real long-horizon problems, and what underPINN's
windowed transfer-learning mode does for unsteady flows — is to march through time in
windows:

1. Split $[0,T]$ into $K$ windows $[0,\Delta T], [\Delta T, 2\Delta T], \dots$
2. Train a PINN on window 1 only, with the true IC.
3. Window $k$'s "IC" is window $k{-}1$'s prediction at the shared edge, and — the
   **transfer-learning step** — window $k$'s network is **initialised with window
   $k{-}1$'s trained weights** instead of from scratch.

Why transfer learning matters here, on three counts:

- **It kills the pathology structurally.** Within one window, $\rho\,\Delta T$ is small:
  the solution barely moves, no steep transition must be threaded in one go, and the
  trivial solution is no longer attractive because the IC dominates a short horizon.
  Causality violation *cannot develop* — no window ever sees a long future.
- **The warm start makes it cheap.** Window $k$'s solution looks like window $k{-}1$'s,
  slightly evolved. Starting from the previous weights, later windows converge in half
  the epochs below (1500 vs 3000) — the network already knows the solution's *features*
  and only nudges them forward.
- **It bounds the blast radius.** Each window needs only local accuracy, like a classical
  time-stepper; a failure can't poison the whole time domain at once.

The honest trade-off: window $k$ inherits window $k{-}1$'s error in its IC, so **error
accumulates across windows** — you've swapped a global optimization pathology for the
familiar, controllable error budget of time-stepping. Usually a very good trade.

In [ ]:
# Cell 4 -- Remedy 2: time-marching with TRANSFER LEARNING (warm starts)
import copy

K  = 4            # number of time windows
dT = 1.0 / K

t0 = time.perf_counter()
models  = []
u_start = h0(x_ic)                         # true IC for window 1
total_epochs = 0
for k in range(K):
    if k == 0:
        model = make_mlp()
        epochs = 3000                      # window 1 trains from scratch
    else:
        model = copy.deepcopy(models[-1])  # TRANSFER: warm-start from previous window
        epochs = 1500                      # warm start needs half the epochs
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    torch.manual_seed(k)                   # this window's collocation points
    xb = torch.rand(2000, device=device) * 2*np.pi
    tb = torch.rand(2000, device=device) * dT + k*dT
    Pw = torch.stack([xb, tb], 1)
    t_ic = torch.full_like(x_ic, k*dT)     # window IC lives at its LEFT edge

    for e in range(epochs):
        opt.zero_grad()
        Pc = Pw.detach().clone().requires_grad_(True)
        u  = model(Pc)
        g  = torch.autograd.grad(u, Pc, torch.ones_like(u), create_graph=True)[0]
        loss = ((g[:, 1:2] - RHO*u*(1-u))**2).mean() \
             + 100*((model(torch.cat([x_ic, t_ic], 1)) - u_start)**2).mean()
        loss.backward(); opt.step()
    total_epochs += epochs
    models.append(model)

    # IC handoff: this window's END state becomes the next window's IC
    with torch.no_grad():
        u_start = model(torch.cat([x_ic, torch.full_like(x_ic, (k+1)*dT)], 1)).detach()
    print(f'window {k+1}/{K}  t in [{k*dT:.2f},{(k+1)*dT:.2f}]  ({epochs} epochs)')

# stitch the piecewise solution together and score it
with torch.no_grad():
    up = torch.zeros_like(Ue); tflat = Pe[:, 1]
    for k in range(K):
        m = (tflat >= k*dT - 1e-9) & (tflat <= (k+1)*dT + 1e-9)
        up[m] = models[k](Pe[m])
    err_tm = torch.sqrt(torch.mean((up - Ue)**2)/torch.mean(Ue**2)).item()
if device.type == 'cuda': torch.cuda.synchronize()
print(f'\ntime-marching + transfer: {time.perf_counter()-t0:.0f} s, '
      f'{total_epochs} total epochs, relative L2 = {err_tm:.4f}')
print(f'compare:  plain = {hist_plain["err"][-1]:.4f} (12000 ep),  '
      f'causal = {hist_causal["err"][-1]:.4f} (12000 ep)')

# final profiles: all three methods vs exact at t = 1
xs = torch.linspace(0, 2*np.pi, 200, device=device).reshape(-1, 1)
tt = torch.full_like(xs, 1.0)
plt.figure(figsize=(9, 4))
plt.plot(xs.cpu(), u_exact(xs, tt).cpu(), 'g', lw=2.4, label='exact @ t=1')
with torch.no_grad():
    plt.plot(xs.cpu(), model_plain(torch.cat([xs, tt], 1)).cpu(),  'b--', label=f'plain ({hist_plain["err"][-1]:.2f})')
    plt.plot(xs.cpu(), model_causal(torch.cat([xs, tt], 1)).cpu(), 'r-.', label=f'causal ({hist_causal["err"][-1]:.2f})')
    plt.plot(xs.cpu(), models[-1](torch.cat([xs, tt], 1)).cpu(),   'm:', lw=2.2, label=f'transfer ({err_tm:.2f})')
plt.xlabel('x'); plt.ylabel('u'); plt.legend(); plt.grid(alpha=.3)
plt.title('Three trainings, one truth: plain vs causal vs time-marching+transfer')
plt.tight_layout(); plt.show()

## Takeaways

- **The trap is an exact solution.** $u\equiv 0$ zeroes the PDE residual *everywhere*; only
  the IC loss objects. The optimizer takes the trade — and the loss curve looks perfectly
  healthy while the error climbs toward 100%. (Second silent failure in this kit: residual
  $\ne$ error.)
- **Both cures do the same thing:** force training to consume time in the order physics
  does. Causal weighting does it *softly* (a weight frontier sweeping forward within one
  run); time-marching + transfer does it *hard* (sequential windows with warm starts).
- **Scoreboard (same problem, verified):** plain ≈ 0.99, causal ≈ 0.08 (12000 epochs),
  time-marching + transfer ≈ 0.11 (7500 total epochs — the warm starts are why it's
  cheaper per window).
- **Transfer learning is the scaling story.** For long-horizon or expensive problems
  (pulsatile/unsteady flows), one giant causal training becomes impractical; windowed
  marching with weight transfer is what underPINN ships for exactly this.

**Experiments to try:** `RHO = 5` (plain limps through — the trap is shallower),
`RHO = 20` (harder for all three); vary `eps` (too small → no causality, too large → the
frontier never advances); vary `K` (more windows = shallower traps but more IC handoffs —
watch the error-accumulation trade-off); disable the warm start (`model = make_mlp()` for
every window) and watch later windows need far more epochs.